# Maxence

In [2]:
import math
import pandas as pd
from collections import Counter

In [38]:
dataset = pd.read_csv('../data/sup100tweet.csv')

# Retrieve first 200 tweets as example
tweets = dataset['tweet']

# Merge all tweets by the same user
user_texts = dataset.groupby('username')['tweet'].agg(lambda x: ' '.join(x)).reset_index()

# Optional: rename columns for clarity
user_texts.columns = ['username', 'merged_tweet']
tweets = user_texts['merged_tweet']


# Simple preprocessing: lowercase, split by space, discarding mentions and URLs as they are too specific
docs = []
clean_tweets = []  
for tweet in tweets:
    if not tweet or tweet.strip() == "":
        continue
    words = tweet.lower().split()
    clean_words = [w for w in words if not w.startswith('@') and not w.startswith('http') and not w.startswith('www') ]
    if clean_words:
        docs.append(clean_words)
        clean_tweets.append(tweet)  # keep the original tweet for reference



# Calculate TF (normalized by max frequency in the document)
tf_list = []
for doc in docs:
    counts = Counter(doc)
    max_count = max(counts.values())
    tf = {word: count/max_count for word, count in counts.items()}
    tf_list.append(tf)

# Calculate IDF (Inverse Document Frequency)
all_words = set(word for doc in docs for word in doc)

N = len(docs)
idf = {}
for word in all_words:
    df = sum(1 for doc in docs if word in doc)
    idf[word] = math.log2(N/df) 

# Calculate TF-IDF
tfidf_list = []
for tf in tf_list:
    tfidf = {word: tf[word]*idf[word] for word in tf}
    tfidf_list.append(tfidf)


top_keywords_per_tweet = []

# Extract top keywords per tweet
for i, tfidf in enumerate(tfidf_list):
    top_keywords = sorted(tfidf.items(), key=lambda x: x[1], reverse=True)[:5]
    # Store as a dict with tweet and its top keywords
    top_keywords_per_tweet.append({
        'tweet': tweets[i],
        'username': user_texts['username'].iloc[i],
        'top_keywords': [(word, score) for word, score in top_keywords],
        'best_TF_IDF': top_keywords[0][1]
    })

# Sort tweets by the best TF-IDF score, descending
top_tweets = sorted(top_keywords_per_tweet, key=lambda x: x['best_TF_IDF'], reverse=True)[:5]

# Print them
for t in top_tweets:
    print(f"username: {t['username']}")
    print(f"Top keywords: {t['top_keywords']} \n")


username: KevinEdwardsJr
Top keywords: [('me..its', 7.330916878114617), ('time..follow', 6.684071271222151), ('laugh..have', 6.511579109384159), ('simple', 3.87148525947732), ('laugh..follow', 0.6468456068924662)] 

username: wowlew
Top keywords: [('isplayer', 7.330916878114617), ('died!', 6.330916878114617), ('has', 0.48542682717024166), ('sorry', 0.48542682717024166), ('isvalcore', 0.06981825598204397)] 

username: twishes
Top keywords: [('[-o]', 6.222038694870389), ('wish', 0.4768314655261952), ('wishes', 0.33673874172956336), ('could', 0.14509914363493986), ('beside', 0.10640196433806078)] 

username: _magic8ball
Top keywords: [('outlook', 6.209168476612413), ('doubtful', 3.101541756125415), ('sources', 2.8195834146594683), ('count', 1.998884712975977), ('likely', 1.0039518875255398)] 

username: lost_dog
Top keywords: [('lost.', 5.745954377393461), ('home.', 3.0089887832272546), ('help', 1.181169758609935), ('find', 1.0089887832272548), ('please', 0.8879733822658887)] 

